# HDAData package documentation

This package serves as a high-dimensional data analysis tool for Python using bioinformatics data, as part of the DAT-M-HIGHDIM lab project.

Contents: 
- [How to install](#-How-to-install)
- [Load Dependencies](#-Load-Dependencies)
- [Tutorial](#-Tutorial)

    The tutorial shows how to use the package's main functionalities:
    1. [Loading files and data structures](##1.-Loading-files-and-data-structures)
    2. [Distance functions for dynamic models and heatmaps](##2.-Distance-functions-for-dynamic-models-and-heatmaps)
    3. Clustering algorithms
    4. Dimensionality reduction algorithms

---

# How to install

1. Make sure you are in the root folder directory where `pyproject.toml` lives. 
2. In your terminal, install the package by running the following command:

    ```bash
    pip install -e .
    ```

# Load Dependencies

After installing the package, make sure to load the necessary dependencies first.

In [1]:
from myhda_ur import Loader, HDAData, plot_heatmap

# Tutorial

This tutorial shows how to use the HDAData package following the provided sample data in the `/data` folder as a guide.

> Background on the sample data

**Scifate data** 
- Sci-fate is a high-throughput sequencing method designed to track transcriptional dynamics of cells (i.e cells which are newly synthesized within a specific window of time) as compared to traditional methods which only observe a "static snapshot" of the cell at a specific moment
- By labelling new mRNA with the 4sU molecule, "old" and "new" genetic activity in the cell can be tracked over a 10-hour response
- Link to [original paper](https://www.nature.com/articles/s41587-020-0480-9) to learn more

<br>

| File name | Description | Information | Dataset size (gene x cell) | 
| --------- | ----------- | ------------ | ------------ |
matrix.mtx.gz | Gene expression matrix | Dataset showing the activity levels of all genes across all individual cells | (43750, 7404) |
4sU.Binom.ntr.mtx.gz | New to Total Ratio matrix | The proportion of a gene's total expression that is "new” | (43750, 7404) |
barcodes.tsv.gz | Barcodes + metadata | Unique molecular tags used to identify cells | (7404, 1) |
features.tsv.gz | Features + metadata | Metadata on genes measured | (43750, 5) |
trajectories_sci.csv | Trajectory table | Trajectory of how cells are connected to each other across time points | (949, 7) |


Throughout this tutorial, the following terms found in the HDAData object would be referred as follows:
- X = gene expression matrix
- ntr = new to total ratio matrix
- obs = barcodes
- var = features

## 1. Loading files and data structures

- [Load files and Create HDAData Object](###-Load-files-and-Create-HDAData-Object)
- [Inspect object](###-Inspect-object)
- [View matrix](###-View-matrix)
- [Slicing](###-Slicing)
- [Gene gene expression](###-Get-gene-expression)

### Load files and Create HDAData Object

In [2]:
# Option 1: Load with default /data directory and initialize the HDAData Object directly

loader = Loader()
scifate = loader.load_as_hda()

In [ ]:
# Option 2: Load with specific folder path and initialize the HDAData object manually 

loader = Loader(dir = "<your_folder_path>")
loaded = loader.load_files()

scifate = HDAData(
    X=loaded["gene_expression_matrix"],
    obs=loaded["barcodes_data"],
    var=loaded["features_data"],
    layers={"ntr": loaded["new_to_total_ratio_matrix"]},
    trajectories=loaded["trajectory_table"]
)

### Inspect object

In [ ]:
# Printing the HDAData object gives you an overview of its elements.

print(scifate)

HDAData object with:
  n_vars (genes): 43750
  n_obs (cells): 7404
  X shape: (43750, 7404)
  layers: ['ntr']
  trajectories: Yes


In [ ]:
# Inspect the Barcodes data (obs).

scifate.obs.head()

,name,cell,time,barcode
cell_name,,,,
A549.4h.AAAAACTCTCTCAA,A549.4h.AAAAACTCTCTCAA,A549,4h,AAAAACTCTCTCAA
A549.6h.AAAAAAGAAGTTCA,A549.6h.AAAAAAGAAGTTCA,A549,6h,AAAAAAGAAGTTCA
A549.8h.AAAAACGGAGAATA,A549.8h.AAAAACGGAGAATA,A549,8h,AAAAACGGAGAATA
A549.4h.AAAACGATCCTGGA,A549.4h.AAAACGATCCTGGA,A549,4h,AAAACGATCCTGGA
A549.6h.AAAAACTGAATTCC,A549.6h.AAAAACTGAATTCC,A549,6h,AAAAACTGAATTCC


In [ ]:
# Inspect the Features data (var).

scifate.var.head()

,gene_id,gene_symbol,feature_type,annotation,length
gene_symbol,,,,,
MTOR-AS1,ENSG00000225602,MTOR-AS1,Gene Expression,Exonic (h.ens90),705
ANGPTL7,ENSG00000171819,ANGPTL7,Gene Expression,Exonic (h.ens90),2238
RPL39P6,ENSG00000238173,RPL39P6,Gene Expression,Exonic (h.ens90),150
SNORA59A,ENSG00000239149,SNORA59A,Gene Expression,Exonic (h.ens90),152
AADACL4,ENSG00000204518,AADACL4,Gene Expression,Exonic (h.ens90),1575


In [ ]:
# Inspect the Trajectories table.

scifate.trajectories.head()

,0h,2h,4h,6h,8h,10h
1,A549.0h.CCGGGCGCTCGACG,A549.2h.GCTGGCGGTTATTG,A549.4h.CTACTGGTCCAGTC,A549.6h.GACAATGCTCGCAA,A549.8h.AAAAAACTACCGAC,A549.10h.AATGCAACCGGATC
2,A549.0h.CGTCCTTAGTTCAA,A549.2h.AGCTACTCCGCCAA,A549.4h.GTTACTCTTAGCGG,A549.6h.AGTCTCGTAGGCTT,A549.8h.AAAAACGGAGAATA,A549.10h.CTCCTGGATCAGGC
3,A549.0h.GGGATAGCTGCTAC,A549.2h.CAGGCGATCATTAC,A549.4h.CAAAGACGAGGTTG,A549.6h.CCTAGATCAGTCAT,A549.8h.AAAATAGAATAGCC,A549.10h.GGCTGACCTGGAAT
4,A549.0h.CGACATAATCTCGC,A549.2h.GCTAACTGGTAGAT,A549.4h.GTTCTCTCTCATCC,A549.6h.GACTGCTCATATGC,A549.8h.AAAATTCTACGCCA,A549.10h.CCGCCTCGGTCAAT
5,A549.0h.CTGTAGGTCTATGG,A549.2h.ATCTATCTAGGTTC,A549.4h.GCGGATCCTACCTC,A549.6h.GCGAGGCCTCCATT,A549.8h.AAACAACTACCGAC,A549.10h.ACCTGGTCCAGGAG


### View matrix

Display matrix (e.g X or ntr) in different formats by first n rows or columns. By default, it displays the gene expression matrix (X). 

Formats supported:
- COO Coordinate List format format (row, column, value) 
    -> returns None if it only contains zero entries
- Dense format -> numeric matrix in numpy arrays
- Labelled view -> labelled gene names as rows and cell names as columns in pandas DataFrame

In [ ]:
# Inspect the first 5 rows and columns of the gene expression matrix (X)

# Option 1: COO
print(f'First 5 rows and columns of X in COO format: ')
print(scifate.show(n_rows=5, n_cols=5, mode="coo"))
print(f'-'*20)

# Option 2: Dense
print(f'\nFirst 5 rows and columns of X in Dense format: ')
print(scifate.show(n_rows=5, n_cols=5, mode="dense"))
print(f'-'*20)

# Option 3: Labelled 
print(f'\nFirst 5 rows and columns of X in Label format: ')
df_X = scifate.show(n_rows=5, n_cols=5, mode="label")
df_X


First 5 rows and columns of X in COO format: 
None
--------------------

First 5 rows and columns of X in Dense format: 
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
--------------------

First 5 rows and columns of X in Label format: 


cell_name,A549.4h.AAAAACTCTCTCAA,A549.6h.AAAAAAGAAGTTCA,A549.8h.AAAAACGGAGAATA,A549.4h.AAAACGATCCTGGA,A549.6h.AAAAACTGAATTCC
gene_symbol,,,,,
MTOR-AS1,0,0,0,0,0
ANGPTL7,0,0,0,0,0
RPL39P6,0,0,0,0,0
SNORA59A,0,0,0,0,0
AADACL4,0,0,0,0,0


### Slicing

Get subset of HDAData using gene or cell names or index.

In [ ]:
# Example indexing patters:

scifate['Gene1', 'Cell1']                      # Single gene and cell
scifate[['Gene1', 'Gene2'], :]                 # Multiple genes, all cells
scifate[:, ['Cell1', 'Cell2']]                 # All genes, specific cells
scifate['Gene1':'Gene10', 'Cell1':'Cell5']     # Slice by name
scifate[:5, ['Cell1', 'Cell2']]                # Slice by gene index and cell names
scifate[['Gene1', 'Gene2'], :5]                # Slice by gene name and cell index
scifate[10:20, 30:40]                          # Integer indexing

In [ ]:
# Example 1: Get cell using cell name
cell1 = scifate[:, "A549.10h.GAGCTGAGACTCTA"]
cell1.show(n_rows=5, mode="label")

cell_name,A549.10h.GAGCTGAGACTCTA
gene_symbol,
MTOR-AS1,1
ANGPTL7,0
RPL39P6,0
SNORA59A,0
AADACL4,0


In [ ]:
# Example 2: Get gene using gene name
gene1 = scifate["MTOR-AS1", :]
gene1.show(n_cols=5, mode="label")

cell_name,A549.4h.AAAAACTCTCTCAA,A549.6h.AAAAAAGAAGTTCA,A549.8h.AAAAACGGAGAATA,A549.4h.AAAACGATCCTGGA,A549.6h.AAAAACTGAATTCC
gene_symbol,,,,,
MTOR-AS1,0,0,0,0,0


In [ ]:
# Example 3: Subset 10 rows and 10 columns from X
subset = scifate[10:20, 30:40]
subset.show(n_rows=5, n_cols=5, mode="label")

cell_name,A549.4h.AAAATCCGTCGCCA,A549.0h.AAAAACGCGTATCG,A549.2h.AAAACAGCAGTATC,A549.0h.AAAACATTATTACC,A549.10h.AAAACGTATTGAGA
gene_symbol,,,,,
PRAMEF12,0,0,0,0,0
PRAMEF20,0,0,0,0,0
AL354712.1,0,0,0,0,0
RPL7P11,0,0,0,0,0
AL357552.1,0,0,0,0,0


### Get gene expression

Query expression value directly for specific gene and cell.

In [ ]:
# Example 1: get gene expression using gene name and cell name
print(scifate.get_expression("MTOR-AS1", "A549.10h.GAGCTGAGACTCTA"))

# Example 2: get gene expression using gene index and cell index
print(scifate.get_expression(0, 0))

1


## 2. Distance functions for dynamic models and heatmaps

This section of the tutorial demonstrates the use of distance functions and heatmaps using an example biological use case on the scifate dataset.

<i><b>Biological Use Case: Identify dynamically regulated transcriptional programs during cortisol stress response using Highly Variable Genes (HVGs)</i></b>

The following similarities heatmaps are explored:

| Analysis | Biological Interpretation |
| --------- | ----------- | 
| Gene-gene similarity | Co-regulated dynamic stress-response genes |
| Cell-cell similarity | Cells with similar transcriptional stress states |
| Trajectory similarity | Similar temporal response paths |

- [Highly Variable Genes (HVG)](###-Highly-Variable-Genes-(HVG))
- [Gene-gene similarity](###-Gene-gene-similarity)
- [Cell-cell similarity](###-Cell-cell-similarity)
- [Trajectory-trajectory similarity](###-Trajectory-trajectory-similarity)

### Highly Variable Genes (HVG)

Since single-cell matrices are highly sparse, using all genes directly for analysis often produces unstable correlations. To obtain an informative subset from the sparse matrix, we first need to identify the Highly Variable Genes (HVG). 

In this tutorial, HVGs are identified using the NTR matrix because NTR captures active transcriptional dynamics, making it suitable for studying temporal stress responses. For each gene, the variance of its NTR values across all cells is computed. Genes with higher variance indicate more active transcriptional changes across cells, such as initiating stress response.

For a gene $g_i$, the variance across all cells is computed as:

$\mathrm{Var}(g_i)=\frac{1}{N}\sum_{j=1}^{N}(x_{ij}-\mu_i)^2$

where:
- $x_{ij}$ is the NTR value of $g_i$ in cell $j$
- $N$ is the total number of cells
- $\mu_i$ is the mean NTR of $g_i$

The variance of the genes are then ranked decreasingly, where the top $n$ genes are selected as HVGs.

In [ ]:
# Get top 10 HVGs

top_10_genes = scifate.get_highly_variable_genes(n=10)
print(f'Top 10 gene names: {top_10_genes}')

Top 10 gene names: ['YOD1', 'SGK1', 'PARD6B', 'SMAD7', 'KLF6', 'KLF10', 'CEBPD', 'PLK2', 'ADM', 'GRPEL2']


In [ ]:
# Initialize a temporary subset object for the top 10 HVGs

subset_top10 = scifate[top_10_genes, :]
subset_top10

HDAData object with:
  n_vars (genes): 10
  n_obs (cells): 7404
  X shape: (10, 7404)
  layers: ['ntr']
  similarities: []
  trajectories: Yes

### Gene-gene similarity 

Gene-gene similarity with raw ntr

In [5]:
scifate.compute_similarity(
    data=subset_top10,
    key="ntr",
    kind="gene",
    method="correlation",
    name="gg_hvg_ntr"
)

HDAData object with:
  n_vars (genes): 43750
  n_obs (cells): 7404
  X shape: (43750, 7404)
  layers: ['ntr']
  similarities: ['gg_hvg_ntr']
  trajectories: Yes

In [6]:
gg_hvg_ntr = scifate.similarities["gg_hvg_ntr"]
gg_hvg_ntr

,YOD1,SGK1,PARD6B,SMAD7,KLF6,KLF10,CEBPD,PLK2,ADM,GRPEL2
YOD1,1.000000,0.100134,0.062567,0.091714,0.169095,0.065951,0.067229,0.136405,0.122610,0.127108
SGK1,0.100134,1.000000,0.036117,0.083644,0.083344,0.053835,0.125137,0.057928,0.049556,0.077367
PARD6B,0.062567,0.036117,1.000000,0.054674,0.085799,0.064356,0.076837,0.049415,0.087599,0.095415
SMAD7,0.091714,0.083644,0.054674,1.000000,0.110295,0.089687,0.027949,0.090426,0.104398,0.093695
KLF6,0.169095,0.083344,0.085799,0.110295,1.000000,0.114303,0.077222,0.138555,0.169065,0.109297
KLF10,0.065951,0.053835,0.064356,0.089687,0.114303,1.000000,0.029685,0.106823,0.099798,0.073749
CEBPD,0.067229,0.125137,0.076837,0.027949,0.077222,0.029685,1.000000,0.006355,0.020968,0.037757
PLK2,0.136405,0.057928,0.049415,0.090426,0.138555,0.106823,0.006355,1.000000,0.192419,0.094190
ADM,0.122610,0.049556,0.087599,0.104398,0.169065,0.099798,0.020968,0.192419,1.000000,0.098048
GRPEL2,0.127108,0.077367,0.095415,0.093695,0.109297,0.073749,0.037757,0.094190,0.098048,1.000000


In [7]:
plot_heatmap(sim=gg_hvg_ntr, type="gene", cluster=True)

Gene-gene similarity with trajectory-based mean ntr

In [8]:
scifate.compute_similarity(
    data=subset_top10,
    key="mean_ntr",
    kind="gene",
    method="correlation",
    name="gg_hvg_mean_ntr"
)

HDAData object with:
  n_vars (genes): 43750
  n_obs (cells): 7404
  X shape: (43750, 7404)
  layers: ['ntr']
  similarities: ['gg_hvg_ntr', 'gg_hvg_mean_ntr']
  trajectories: Yes

In [9]:
gg_hvg_mean_ntr = scifate.similarities["gg_hvg_mean_ntr"]
gg_hvg_mean_ntr 

,YOD1,SGK1,PARD6B,SMAD7,KLF6,KLF10,CEBPD,PLK2,ADM,GRPEL2
YOD1,1.000000,0.776945,0.474773,0.454065,0.556404,0.353280,0.735725,-0.311807,0.546889,-0.533654
SGK1,0.776945,1.000000,0.700703,0.057479,0.799484,-0.122128,0.910030,-0.750463,0.299914,-0.799362
PARD6B,0.474773,0.700703,1.000000,-0.464021,0.744717,-0.636363,0.909213,-0.917815,-0.339614,-0.313180
SMAD7,0.454065,0.057479,-0.464021,1.000000,-0.102760,0.935665,-0.148956,0.575593,0.928657,-0.376392
KLF6,0.556404,0.799484,0.744717,-0.102760,1.000000,-0.350402,0.886445,-0.786754,0.179731,-0.564354
KLF10,0.353280,-0.122128,-0.636363,0.935665,-0.350402,1.000000,-0.347462,0.744392,0.826952,-0.129432
CEBPD,0.735725,0.910030,0.909213,-0.148956,0.886445,-0.347462,1.000000,-0.861710,0.054189,-0.569163
PLK2,-0.311807,-0.750463,-0.917815,0.575593,-0.786754,0.744392,-0.861710,1.000000,0.348633,0.431287
ADM,0.546889,0.299914,-0.339614,0.928657,0.179731,0.826952,0.054189,0.348633,1.000000,-0.557876
GRPEL2,-0.533654,-0.799362,-0.313180,-0.376392,-0.564354,-0.129432,-0.569163,0.431287,-0.557876,1.000000


In [10]:
plot_heatmap(sim=gg_hvg_mean_ntr, type="gene", cluster=True)

### Cell-cell similarity

Cell-cell similarity with normalized expression values

In [12]:
cells_hvg = scifate.get_timepoint_cells(n=2)
cells_hvg

['A549.0h.CGTTACGGCAACCA',
 'A549.0h.ACGCCTCGATGGAG',
 'A549.2h.GAGTCCTACGGCAA',
 'A549.2h.CCTTCCTAAGCGGT',
 'A549.4h.AGTTGTCGAGGTAT',
 'A549.4h.GACGATGAGTTCTC',
 'A549.6h.ATATTCCTTACCAA',
 'A549.6h.GCCTAATCGAACTC',
 'A549.8h.AAGGCGGCAGAGGA',
 'A549.8h.CCGAGACCGTAGCA',
 'A549.10h.GATTATCGTATCAA',
 'A549.10h.GAAAGACCTGGAAT']

In [13]:
subset_hvg_cell = scifate[top_10_genes, cells_hvg]
subset_hvg_cell

HDAData object with:
  n_vars (genes): 10
  n_obs (cells): 12
  X shape: (10, 12)
  layers: ['ntr']
  similarities: []
  trajectories: Yes

In [14]:
scifate.compute_similarity(
    data=subset_hvg_cell,
    key="X",
    kind="cell",
    method="correlation",
    name="cc_hvg_X"
)

HDAData object with:
  n_vars (genes): 43750
  n_obs (cells): 7404
  X shape: (43750, 7404)
  layers: ['ntr']
  similarities: ['gg_hvg_ntr', 'gg_hvg_mean_ntr', 'cc_hvg_X']
  trajectories: Yes

In [15]:
cc_hvg_X = scifate.similarities["cc_hvg_X"]
cc_hvg_X.head()

,A549.0h.CGTTACGGCAACCA,A549.0h.ACGCCTCGATGGAG,A549.2h.GAGTCCTACGGCAA,A549.2h.CCTTCCTAAGCGGT,A549.4h.AGTTGTCGAGGTAT,A549.4h.GACGATGAGTTCTC,A549.6h.ATATTCCTTACCAA,A549.6h.GCCTAATCGAACTC,A549.8h.AAGGCGGCAGAGGA,A549.8h.CCGAGACCGTAGCA,A549.10h.GATTATCGTATCAA,A549.10h.GAAAGACCTGGAAT
A549.0h.CGTTACGGCAACCA,1.000000,0.557330,-0.163137,-0.767532,0.355452,0.023286,-0.419661,0.437261,0.692732,-0.041323,-0.118096,0.580651
A549.0h.ACGCCTCGATGGAG,0.557330,1.000000,0.287607,-0.398935,0.300332,-0.203275,-0.338106,-0.085964,0.501043,0.372256,0.156860,0.633820
A549.2h.GAGTCCTACGGCAA,-0.163137,0.287607,1.000000,0.181268,0.202045,-0.345880,0.265429,-0.227486,-0.454090,0.575101,0.088732,0.159971
A549.2h.CCTTCCTAAGCGGT,-0.767532,-0.398935,0.181268,1.000000,-0.171060,0.218254,0.507829,-0.303359,-0.370632,0.246456,-0.020914,-0.443681
A549.4h.AGTTGTCGAGGTAT,0.355452,0.300332,0.202045,-0.171060,1.000000,0.193531,-0.555189,0.393457,0.415277,-0.162279,-0.062630,0.583338


In [16]:
plot_heatmap(sim=cc_hvg_X, type="cell", cluster=True)

### Trajectory-trajectory similarity

Trajectory-trajectory similarity with ntr based on HVGs

In [17]:
top_10_traj = subset_top10.get_highly_dynamic_trajectories(n=10)
top_10_traj

[884, 194, 157, 797, 926, 160, 252, 709, 349, 866]

In [18]:
scifate.compute_similarity(
    data=scifate,
    key="ntr",
    kind="trajectory",
    trajectory_id=top_10_traj,
    method="correlation",
    name="tt_hvg_ntr"
)

HDAData object with:
  n_vars (genes): 43750
  n_obs (cells): 7404
  X shape: (43750, 7404)
  layers: ['ntr']
  similarities: ['gg_hvg_ntr', 'gg_hvg_mean_ntr', 'cc_hvg_X', 'tt_hvg_ntr']
  trajectories: Yes

In [19]:
tt_hvg_ntr = scifate.similarities["tt_hvg_ntr"]
tt_hvg_ntr.head()

,traj_884,traj_194,traj_157,traj_797,traj_926,traj_160,traj_252,traj_709,traj_349,traj_866
traj_884,1.000000,0.202188,0.199536,0.214475,0.180266,0.193247,0.193123,0.191457,0.191993,0.201414
traj_194,0.202188,1.000000,0.182796,0.202825,0.178163,0.206497,0.170755,0.188451,0.194947,0.188505
traj_157,0.199536,0.182796,1.000000,0.192822,0.174866,0.173306,0.180421,0.184718,0.174145,0.190824
traj_797,0.214475,0.202825,0.192822,1.000000,0.177453,0.193757,0.180179,0.192785,0.185535,0.199943
traj_926,0.180266,0.178163,0.174866,0.177453,1.000000,0.167112,0.166484,0.164525,0.158190,0.166622


In [20]:
plot_heatmap(sim=tt_hvg_ntr, type="trajectories", cluster=False)